## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html)  for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy matplotlib

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["gurobi", "xpress", "highs", "gokestrel"], # modules to install
    license_uuid="default") # license to use

### AMPL Formulation
#### Model Fragment

In [ ]:
%%ampl_eval
reset;           # Clears all AMPL model components, data, and options

# Base sets
set PROD;                      # products
set NODES;                     # nodes

# Parameters linking products to nodes
set ORIG {PROD} within NODES;   # origin node per product: 
                                # If ORIG[p] is not in NODES, AMPL rejects the data because it must be within NODES.
set DEST {PROD} within NODES;   # destination node per product: 
                                # If DEST[p] is not in NODES, AMPL rejects the data because it must be within NODES.

# Global arc set
set ARCS dimen 2 within NODES cross NODES;  # valid arcs (i,j).
                                            # Each arc must have exactly 2 components.
                                            # Both components must be members of NODES.

# 1. Minimal indexed set
set AREA {p in PROD};   # set per product
                        # There is no "within ..." restriction here.
                        # So AREA[p] can contain almost any symbolic values.         

# 2. Indexed subset of arcs
set OUT_ARCS {i in NODES} within ARCS;  # subset of valid arcs assigned to node i;
                                        # AMPL checks membership in ARCS, but does not automatically check that the arc starts at i

# 3. Symbolically defined set (per product)
set LINKS {p in PROD} = ORIG[p] cross DEST[p];  # LINKS[p] is calculated automatically by AMPL.
                                                # It is the Cartesian product of ORIG[p] and DEST[p].

# 4. Default empty sets
set AREA2 {p in PROD} default {};   # AREA2[p] is empty unless you give data for it.

# 5. Multi-dimensional indexing
set S {i in NODES} within PROD; # S[i] is a set of products associated with node i. The index must be from NODES. 

#### Data

In [ ]:
%%ampl_eval
reset data;             # Clears all loaded data values but keeps the model declarations.
data;                   # Start data mode

# --- Base sets ---
set PROD := P1 P2 P3;
set NODES := A B C D;

# --- Set-valued ORIG/DEST (allowing multiple nodes per product) ---
set ORIG["P1"] := A, B ;
set ORIG["P2"] := B, C ;
set ORIG["P3"] := C, D ;

set DEST["P1"] := B ;
set DEST["P2"] := C ;
set DEST["P3"] := C, D;

# --- Global arcs ---
set ARCS := (A,B), (A,C), (B,C), (B,D), (C,A), (C,D), (D,A);

# --- 1. AREA (indexed set per product) ---
set AREA["P1"] := AAA, BBB ;
set AREA["P2"] := BBB, CCC ;
set AREA["P3"] := CCC, DDD;

# --- 2. OUT_ARCS (subset of ARCS per node) ---
set OUT_ARCS ['A']:= (A,B) (A,C);
set OUT_ARCS ['B']:= (B,C) (B,D);
set OUT_ARCS ['C']:= (C,A) (C,D);
set OUT_ARCS ['D']:= (D,A);

# --- 4. AREA2 (default empty, but override one case) ---
set AREA2['P2'] := B;

# --- 5. S (products per arc) ---
set S['A'] := P1, P2 ;

display PROD, NODES, 
        ORIG, DEST, ARCS, 
        AREA, OUT_ARCS, 
        LINKS, AREA2, S;

set PROD := P1 P2 P3;

set NODES := A B C D;

set ORIG[P1] := A B;
set ORIG[P2] := B C;
set ORIG[P3] := C D;

set DEST[P1] := B;
set DEST[P2] := C;
set DEST[P3] := C D;

set ARCS := (A,B) (A,C) (B,C) (B,D) (C,A) (C,D) (D,A);

set AREA[P1] := AAA BBB;
set AREA[P2] := BBB CCC;
set AREA[P3] := CCC DDD;

set OUT_ARCS[A] := (A,B) (A,C);
set OUT_ARCS[B] := (B,C) (B,D);
set OUT_ARCS[C] := (C,A) (C,D);
set OUT_ARCS[D] := (D,A);

set LINKS[P1] := (A,B) (B,B);
set LINKS[P2] := (B,C) (C,C);
set LINKS[P3] := (C,C) (C,D) (D,C) (D,D);

set AREA2[P2] := B;

set S[A] := P1 P2;



### amplpy Data Loading
This section reuses the ***AMPL model fragment*** declared above and loads the same data from Python.
#### Data

In [93]:
ampl.eval("reset data;")    # Clear only data, keep model declarations

data = {                    # Data for indexed AMPL sets
    "ORIG": {               # origin nodes per product
        "P1": ["A", "B"],
        "P2": ["B", "C"],
        "P3": ["C", "D"],},
    "DEST": {               # destination nodes per product
        "P1": ["B"],
        "P2": ["C"],
        "P3": ["C", "D"],},
    "AREA": {               # area values per product
        "P1": ["AAA", "BBB"],
        "P2": ["BBB", "CCC"],
        "P3": ["CCC", "DDD"],},
    "OUT_ARCS": {           # arcs assigned to each node
        "A": [("A", "B"), ("A", "C")],
        "B": [("B", "C"), ("B", "D")],
        "C": [("C", "A"), ("C", "D")],
        "D": [("D", "A")],},
    "AREA2": {              # optional set; others stay empty
        "P2": ["B"],},
    "S": {                  # products associated with node A
        "A": ["P1", "P2"],},}

# Load base sets
ampl.get_set("PROD").set_values(["P1", "P2", "P3"])
ampl.get_set("NODES").set_values(["A", "B", "C", "D"])

ampl.get_set("ARCS").set_values([   # Load 2-dimensional arc set
    ("A", "B"), ("A", "C"),
    ("B", "C"), ("B", "D"),
    ("C", "A"), ("C", "D"),
    ("D", "A"),])

# Load all indexed sets from dictionary
for set_name, indexed_values in data.items():
    s = ampl.get_set(set_name)
    for idx, vals in indexed_values.items():
        s[idx].set_values(vals)

# Display loaded and computed data
ampl.eval("""
display PROD, NODES,
        ORIG, DEST, ARCS,
        AREA, OUT_ARCS,
        LINKS, AREA2, S;""")

set PROD := P1 P2 P3;

set NODES := A B C D;

set ORIG[P1] := A B;
set ORIG[P2] := B C;
set ORIG[P3] := C D;

set DEST[P1] := B;
set DEST[P2] := C;
set DEST[P3] := C D;

set ARCS := (A,B) (A,C) (B,C) (B,D) (C,A) (C,D) (D,A);

set AREA[P1] := AAA BBB;
set AREA[P2] := BBB CCC;
set AREA[P3] := CCC DDD;

set OUT_ARCS[A] := (A,B) (A,C);
set OUT_ARCS[B] := (B,C) (B,D);
set OUT_ARCS[C] := (C,A) (C,D);
set OUT_ARCS[D] := (D,A);

set LINKS[P1] := (A,B) (B,B);
set LINKS[P2] := (B,C) (C,C);
set LINKS[P3] := (C,C) (C,D) (D,C) (D,D);

set AREA2[P2] := B;

set S[A] := P1 P2;

